# Six Ways to Evaluate a Policy in R — Colab Notebook

R notebook companion to the tutorial [*Six Ways to Evaluate a Policy using R: Comparative Case Studies of Proposition 99*](https://carlos-mendez.org/post/r_causalpolicy_workshop/). Same six estimators applied to California's 1988 Proposition 99 cigarette tax: naive pre-post, Difference-in-Differences, two flavours of Interrupted Time Series, Regression Discontinuity on time, Synthetic Control, and CausalImpact.

**How to run.** In Colab: *Runtime → Change runtime type → R* (not Python). Then *Runtime → Run all*. The package-install cell takes about 5 minutes on a fresh runtime.

**Authoritative copy.** The full tutorial — with equations, common-pitfall callouts, Mermaid diagrams, and discussion — lives at <https://carlos-mendez.org/post/r_causalpolicy_workshop/>. This notebook is a streamlined runnable version of the code blocks.

**Estimand throughout.** The Average Treatment effect on the Treated (ATT) on California, for the post-period 1989--2000.

## 1. Setup and packages

We need nine R packages. Several of them are not in Colab's default R runtime, so the install cell takes a few minutes the first time. The install is idempotent: re-running it on a session that already has the packages is a no-op.

In [ ]:
# Install everything we need. Idempotent: only installs missing packages.
required <- c(
  "pacman",       # convenience loader
  "tidyverse",    # data manipulation + ggplot2
  "sandwich",     # HAC variance estimator
  "lmtest",       # coeftest for HAC-robust inference
  "tidysynth",    # synthetic control (tidy API)
  "fpp3",         # forecasting (tsibble, fable, ARIMA)
  "mice",         # multiple imputation
  "randomForest", # backend for mice method = 'rf'
  "CausalImpact", # Bayesian structural time series
  "broom",        # tidy model output
  "glue"          # string interpolation
)
to_install <- setdiff(required, rownames(installed.packages()))
if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}
cat("All required packages are installed.\n")

In [ ]:
# Load everything and fix the random seed for reproducibility.
pacman::p_load(
  tidyverse, sandwich, lmtest, tidysynth, fpp3,
  mice, CausalImpact, broom, glue
)

set.seed(42)

# Make ggplot output a bit bigger in Colab.
options(repr.plot.width = 10, repr.plot.height = 6)

## 2. The potential outcomes framework (recap)

Every method here estimates $Y_{1t}(0)$ — what California's cigarette sales *would have been* without Proposition 99 — and subtracts that estimate from the observed series. The methods differ in *how* they build that counterfactual:

- From California's own past: Naive pre-post, Interrupted Time Series, RDD on time.
- From a single comparison state: Difference-in-Differences.
- From a weighted blend of donor states: Synthetic Control, CausalImpact.

See [section 2 of the full post](https://carlos-mendez.org/post/r_causalpolicy_workshop/#2-the-potential-outcomes-framework-causal-inference-as-a-missing-data-problem) for the formal $Y(0)$ / $Y(1)$ notation, the missing-data table, and the ATT definition.

## 3. Data: download and inspect

We pull `proposition99.rds` from the project's GitHub repository so the notebook is self-contained on a fresh Colab runtime.

In [ ]:
DATA_URL  <- "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/content/post/r_causalpolicy_workshop/proposition99.rds"
CACHE_RDS <- "proposition99.rds"

if (!file.exists(CACHE_RDS)) {
  download.file(DATA_URL, destfile = CACHE_RDS, mode = "wb")
}

prop99 <- read_rds(CACHE_RDS) |> as_tibble()
glimpse(prop99)

In [ ]:
# California pre/post descriptives -- the puzzle that motivates the rest.
prop99_cali <- prop99 |>
  filter(state == "California") |>
  mutate(prepost = factor(year > 1988, labels = c("Pre", "Post")))

prop99_cali |>
  group_by(prepost) |>
  summarize(n            = n(),
            mean_cigsale = mean(cigsale),
            sd_cigsale   = sd(cigsale),
            .groups = "drop")

In [ ]:
# All 39 states' cigsale 1970-2000, California highlighted in warm orange.
prop99 |>
  mutate(unit_type = if_else(state == "California",
                             "California (treated)", "Donor state")) |>
  ggplot(aes(x = year, y = cigsale, group = state,
             color = unit_type,
             linewidth = unit_type, alpha = unit_type)) +
  geom_line() +
  geom_vline(xintercept = 1988.5, color = "#d97757",
             linetype = "dashed", linewidth = 0.7) +
  scale_color_manual(values = c("California (treated)" = "#d97757",
                                "Donor state" = "#6a9bcc")) +
  scale_linewidth_manual(values = c("California (treated)" = 1.2,
                                    "Donor state" = 0.45)) +
  scale_alpha_manual(values = c("California (treated)" = 1.0,
                                "Donor state" = 0.45)) +
  labs(title = "Per-capita cigarette sales, 1970-2000",
       subtitle = "California in orange; 38 donor states in blue",
       x = "Year", y = "Cigarette sales (packs per capita)",
       color = NULL) +
  theme_minimal() +
  guides(linewidth = "none", alpha = "none")

## 4. Method 1 — Naive pre-post comparison

Compare California's pre- vs post-1989 means. Implicit counterfactual: California's pre-period level continues unchanged. Almost certainly biased downward by the nationwide secular decline in smoking.

In [ ]:
# OLS on California's 1984-1993 window, HAC-robust SEs.
fit_prepost <- lm(cigsale ~ prepost,
                  data = prop99_cali |> filter(year > 1983, year < 1994))
coeftest(fit_prepost, vcov. = vcovHAC)

## 5. Method 2 — Difference-in-Differences (California vs Nevada)

DiD with Nevada as the single control. Identifying assumption: California and Nevada would have moved on parallel paths without Proposition 99. The interaction coefficient `stateCalifornia:prepostPost` is the DiD estimate.

In [ ]:
# Keep CA + NV in the 1984-1993 window. Nevada is the base level so the
# stateCalifornia interaction lands on California's extra change.
prop99_did <- prop99 |>
  filter(state %in% c("California", "Nevada"),
         year > 1983, year < 1994) |>
  mutate(prepost = factor(year > 1988, labels = c("Pre", "Post")),
         state   = factor(state, levels = c("Nevada", "California")))

fit_did <- lm(cigsale ~ state * prepost, data = prop99_did)
coeftest(fit_did, vcov. = vcovHAC)

## 6. Method 3a — Interrupted Time Series via pre-period growth curve

Linear pre-period fit, extrapolated forward. The ATT is the average per-year gap between observed and extrapolated values.

In [ ]:
# California-only series as a tsibble (required by fpp3 below).
prop99_ts <- prop99 |>
  filter(state == "California") |>
  select(year, cigsale) |>
  mutate(prepost = factor(year > 1988, labels = c("Pre", "Post"))) |>
  as_tsibble(index = year) |>
  mutate(year0 = year - 1989)

# Pre-period linear trend on 1970-1988 California cigsale.
fit_growth <- lm(cigsale ~ year, data = prop99_ts |> filter(prepost == "Pre"))
summary(fit_growth)$coefficients

# Extrapolate forward, average the gap with the observed post-period.
post_df <- prop99_ts |> filter(prepost == "Post")
pred_growth <- predict(fit_growth, newdata = as_tibble(post_df))
its_growth_estimate <- mean(post_df$cigsale - pred_growth)
cat("ITS growth-curve ATT estimate:", round(its_growth_estimate, 2), "packs/capita\n")

## 7. Method 3b — Interrupted Time Series via auto-selected ARIMA forecast

AICc-selected ARIMA on the pre-period, forecast 12 years forward as the counterfactual. Spoiler: this is the methodological outlier — the model over-extrapolates the late-1980s acceleration and flips the sign.

In [ ]:
# Fit an ARIMA on 1970-1988 California cigsale; ic = 'aicc' searches over
# (p, d, q) and picks the AICc minimiser.
fit_arima <- prop99_ts |>
  filter(prepost == "Pre") |>
  model(timeseries = ARIMA(cigsale, ic = "aicc"))
report(fit_arima)

# Project 12 years forward as the counterfactual and average the gap.
fcasts <- forecast(fit_arima, h = "12 years")
ce_arima <- post_df$cigsale - fcasts$.mean
cat("ITS ARIMA ATT estimate:", round(mean(ce_arima), 2), "packs/capita\n")

## 8. Method 4 — Regression Discontinuity on time

Piecewise linear regression with a level break and a slope change at the 1989 threshold. The coefficient on `prepostPost` is the level break — the immediate jump at the policy threshold.

In [ ]:
# Piecewise OLS with year0 = year - 1989 as the recentred running variable.
fit_rdd <- lm(cigsale ~ year0 + prepost + year0:prepost,
              data = as_tibble(prop99_ts))
coeftest(fit_rdd, vcov. = vcovHAC)

## 9. Method 5 — Synthetic Control

The headline method. Build a weighted blend of donor states that mimics California's pre-period, then read off the post-period gap. Uses the [`tidysynth`](https://github.com/edunford/tidysynth) package.

In [ ]:
prop99_syn <- prop99 |>
  # 1. Declare the panel structure: outcome, unit, time, treated unit,
  #    intervention year. generate_placebos = TRUE fits the model
  #    treating each donor as if treated, for the permutation test below.
  synthetic_control(
    outcome = cigsale, unit = state, time = year,
    i_unit = "California", i_time = 1988,
    generate_placebos = TRUE
  ) |>
  # 2. Predictors averaged over the full pre-period.
  generate_predictor(
    time_window = 1980:1988,
    lnincome  = mean(lnincome, na.rm = TRUE),
    retprice  = mean(retprice, na.rm = TRUE),
    age15to24 = mean(age15to24, na.rm = TRUE)
  ) |>
  generate_predictor(time_window = 1984:1988,
                     beer = mean(beer, na.rm = TRUE)) |>
  # 2b. Three lagged outcomes pin synthetic California's pre-period trajectory.
  generate_predictor(time_window = 1975, cigsale_1975 = cigsale) |>
  generate_predictor(time_window = 1980, cigsale_1980 = cigsale) |>
  generate_predictor(time_window = 1988, cigsale_1988 = cigsale) |>
  # 3. Solve the constrained QP for donor weights w*.
  generate_weights(optimization_window = 1970:1988) |>
  # 4. Compute the synthetic California series from w* and donor outcomes.
  generate_control()

# Average post-period gap = the Synthetic Control ATT.
sc_post <- grab_synthetic_control(prop99_syn) |>
  filter(time_unit > 1988) |>
  mutate(dif = real_y - synth_y)
cat("Synthetic Control ATT estimate:", round(mean(sc_post$dif), 2), "packs/capita\n")

In [ ]:
# Donor weights (W) -- which states matter, and how much.
grab_unit_weights(prop99_syn) |> arrange(desc(weight)) |> head(8)

In [ ]:
# Predictor weights (V matrix) -- how much weight the optimiser put on
# each pre-period predictor when matching California.
grab_predictor_weights(prop99_syn)

In [ ]:
# tidysynth's built-in plotting helpers all return ggplot objects.
plot_trends(prop99_syn)

In [ ]:
plot_weights(prop99_syn)

In [ ]:
# Placebo permutation test: where does California sit relative to the
# distribution of placebo effects?
plot_placebos(prop99_syn)

In [ ]:
# Fisher exact p-value via the post/pre MSPE ratio.
grab_significance(prop99_syn) |>
  arrange(desc(mspe_ratio)) |>
  head(5)

In [ ]:
plot_mspe_ratio(prop99_syn)

## 10. Method 6 — CausalImpact

Bayesian structural time-series model with donor states as predictors. The most uncertainty-honest estimator in the set: it returns a posterior credible interval rather than a frequentist confidence band.

In [ ]:
# Random-forest multiple imputation for missing covariate values.
prop99_imputed <- prop99 |>
  mice(m = 1, method = "rf", printFlag = FALSE) |>
  complete() |> as_tibble()

# CausalImpact wants a WIDE dataset with the treated outcome in column 1.
prop99_wide <- prop99_imputed |>
  pivot_wider(names_from  = state,
              values_from = c(cigsale, lnincome, beer, age15to24, retprice)) |>
  relocate(cigsale_California) |>
  select(-year)

# Period indices are row numbers, not years.
# Row 1 = 1970, row 19 = 1988 (pre-period); row 20 = 1989, row 31 = 2000.
pre_idx  <- c(1, 19)
post_idx <- c(20, 31)

# Reset the seed for the BSTS MCMC sampler.
set.seed(42)

impact_full <- CausalImpact(prop99_wide,
                            pre.period  = pre_idx,
                            post.period = post_idx)
summary(impact_full)

In [ ]:
# Two-panel diagnostic: observed vs Bayesian counterfactual + cumulative effect.
plot(impact_full)

## 11. Cross-method comparison

All seven estimates side-by-side. The `principled_inference` column records each method's *recommended* uncertainty quantification — which is a frequentist confidence interval for some methods, a Fisher exact $p$-value for Synthetic Control, and a posterior credible interval for CausalImpact.

Five of the six causal methods land in a $-13$ to $-20$ pack consensus. DiD vs Nevada collapses to noise (Nevada was falling in parallel). ITS-ARIMA flips sign (AICc over-extrapolates the late-1980s acceleration).

In [ ]:
results_tbl <- tibble(
  method = c("Naive pre-post", "DiD (CA vs Nevada)", "ITS (growth curve)",
             "ITS (ARIMA)", "RDD on time", "Synthetic Control", "CausalImpact"),
  estimate = c(-27.02, -5.68, -28.28, 4.55, -20.06, -18.72, -12.82),
  principled_inference = c(
    "HAC 95% CI: [-37.4, -16.6]",
    "HAC 95% CI: [-16.3, +4.9]",
    "Linear-trend 95% prediction interval",
    "ARIMA 95% forecast-band average: [-29.1, +38.2]",
    "HAC 95% CI: [-31.0, -9.1]",
    "Fisher exact p = 0.026 (MSPE-ratio rank 1/39)",
    "Posterior 95% CrI: [-31.9, +5.7]; P(effect != 0) = 92%"
  )
)
print(results_tbl, n = Inf, width = Inf)

In [ ]:
# Quick forest plot of the seven estimates.
results_tbl |>
  mutate(method = factor(method, levels = rev(method))) |>
  ggplot(aes(x = estimate, y = method)) +
  geom_vline(xintercept = 0, color = "grey60", linewidth = 0.4) +
  geom_point(size = 3.4, color = "#d97757") +
  geom_text(aes(label = sprintf("%.1f", estimate)),
            vjust = -1.1, color = "#141413", size = 3.2) +
  labs(title = "Six estimators of California's Proposition 99 effect",
       subtitle = "Effect on per-capita cigarette sales (packs). Negative = reduction.",
       x = "Effect on cigarette sales", y = NULL) +
  theme_minimal()

## Wrap-up

Five of the six causal methods agree on a $-13$ to $-20$ pack reduction per year over 1989--2000. The naive pre-post and linear-trend Interrupted Time Series overshoot by roughly 50% because they ignore the nationwide secular decline in smoking. Difference-in-Differences against Nevada collapses to noise (Nevada was falling in parallel). AICc-selected ARIMA flips sign because the model over-extrapolates the late-1980s acceleration.

**Read more.** The full tutorial — with the formal potential-outcomes equations behind each estimator, the common-pitfall callouts, the method-picker decision tree, the Mermaid diagrams, and the discussion — is at <https://carlos-mendez.org/post/r_causalpolicy_workshop/>.

**Go deeper.** [Difference-in-Differences with staggered adoption](https://carlos-mendez.org/post/r_did/) and [Bayesian spatial Synthetic Control on Proposition 99](https://carlos-mendez.org/post/r_sc_bayes_spatial/) are sister tutorials that take individual methods much further.